In [6]:
import json

dictionary = {
    "FACULTY": "faculty",
    "DEPARTMENT": "department",
    "PROGRAM": "program",
    "COURSE_TITLE": "course-title",
}

entity_types = []

def sort(e):
    return len(e[0])

for entity_type in dictionary:
    file_name = dictionary[entity_type]
    file = open(f"../../data/{file_name}.jsonlines", "r")
    lines = file.readlines()

    for line in lines:
        title_ = json.loads(line).get("title")

        if(entity_type == "COURSE_TITLE" and "(" in title_):
            continue
        elif(entity_type == "FACULTY" and "of" not in title_):
            title = "Faculty of " + title_
        elif(entity_type == "DEPARTMENT"):
            title = "Department of " + title_
        elif(entity_type == "PROGRAM"):
            title = title_ + " Porgram"
        else:
            title = title_

        entity_types.append({"label": entity_type, "pattern": title})
    
    file.close()



In [7]:
import spacy

nlp = spacy.blank("en")

entity_ruler = nlp.add_pipe("entity_ruler")
entity_ruler.add_patterns(entity_types)

print(nlp.pipe_names)

nlp.to_disk("../nero_entity_ruler_model")


['entity_ruler']


In [8]:
if "ner" not in nlp.pipe_names:
    ner = nlp.create_pipe("ner")
    nlp.add_pipe("ner", last=True)

print(nlp.pipe_names)


['entity_ruler', 'ner']


In [9]:
import json

file = open("training_data.jsonlines", "r")
lines = file.readlines()

data = []

for line in lines:
    line = json.loads(line)
    data.append(line)


# Add labels
for text, annotations in data:
    for ent in annotations.get("entities"):
        ner.add_label(ent[2])

print(ner.labels)

('COURSE_TITLE', 'DEPARTMENT', 'FACULTY', 'PROGRAM')


In [10]:
import random
from spacy.training.example import Example

optimizer = nlp.begin_training()

training_set = data

for iterate in range(30):
    print("Iteration", iterate)

    random.shuffle(training_set)

    losses = {}

    for text, annotations in training_set:
        doc = nlp.make_doc(text)
        example = Example.from_dict(doc, annotations)
        nlp.update([example], drop=0.2, sgd=optimizer, losses=losses)

    print(losses)

nlp.to_disk("../nero_ner_model")


Iteration 0
{'ner': 160.32631080447425}
Iteration 1
{'ner': 54.25577932819631}
Iteration 2
{'ner': 56.920408854500636}
Iteration 3
{'ner': 58.29468520023066}
Iteration 4
{'ner': 47.456099546041}
Iteration 5
{'ner': 42.56652860748594}
Iteration 6
{'ner': 48.01773293310254}
Iteration 7
{'ner': 23.7365340400772}
Iteration 8
{'ner': 35.13494280681325}
Iteration 9
{'ner': 18.871144772799916}
Iteration 10
{'ner': 25.539433487693582}
Iteration 11
{'ner': 21.05840799862245}
Iteration 12
{'ner': 17.009313764692866}
Iteration 13
{'ner': 18.47063504440063}
Iteration 14
{'ner': 21.830727523124697}
Iteration 15
{'ner': 14.215847332046426}
Iteration 16
{'ner': 16.850069228391266}
Iteration 17
{'ner': 20.736466054864472}
Iteration 18
{'ner': 19.986628786024934}
Iteration 19
{'ner': 9.25983478320457}
Iteration 20
{'ner': 9.693690034027755}
Iteration 21
{'ner': 15.492404569027524}
Iteration 22
{'ner': 13.275337970777445}
Iteration 23
{'ner': 10.526060472788112}
Iteration 24
{'ner': 9.214489296127407}
I